In [94]:
import albumentations as A        # For data augmentation
import cv2                        # For loading images
import matplotlib.pyplot as plt   # For plotting images
import numpy as np
import os
from tqdm import tqdm

In [19]:
def show_augmented(augmentation, image, bbox):
  augmented = augmentation(image=image, bboxes=[bbox], field_id=['1'])
  show_image(augmented['image'], augmented['bboxes'][0])

In [20]:
def show_image(image, bbox):
  image = visualize_bbox(image.copy(), bbox)
  f = plt.figure(figsize=(18, 12))
  plt.imshow(
    cv2.cvtColor(image, cv2.COLOR_BGR2RGB),
    interpolation='nearest'
  )
  plt.axis('off')
  f.tight_layout()
  plt.show()

In [21]:
BOX_COLOR = (255, 0, 0)
def visualize_bbox(img, bbox, color=BOX_COLOR, thickness=2):
    x_min, y_min, x_max, y_max = map(lambda v: int(v), bbox)
    cv2.rectangle(
      img,
      (x_min, y_min),
      (x_max, y_max),
      color=color,
      thickness=thickness
    )
    return img

In [22]:
bbox_params = A.BboxParams(
  format='yolo',
  min_area=1,
  min_visibility=0.5,
  label_fields=['field_id']
)

In [124]:
doc_aug = A.Compose([
    A.Flip(p=0.25),
    A.RandomGamma(gamma_limit=(20, 300), p=0.5),
    A.RandomBrightnessContrast(p=0.85),
    A.Rotate(limit=35, p=0.9),
    A.RandomRotate90(p=0.25),
    A.RGBShift(p=0.75),
    A.GaussNoise(p=0.5),
    A.Blur(p=0.4),
    A.ColorJitter(p=0.3)
], bbox_params=bbox_params)

In [125]:
augmented['field_id']

[1, 2, 2, 2, 2, 2, 1, 1, 1, 1]

In [122]:
class_label

1

In [123]:
import os
import cv2
import pandas as pd
from tqdm import tqdm
import albumentations as A

DATASET_PATH = 'deneme'
IMAGES_PATH = 'test'
os.makedirs(DATASET_PATH, exist_ok=True)
os.makedirs(IMAGES_PATH, exist_ok=True)

# Assuming your text files have the same name as the images but with a '.txt' extension
txt_files = [f for f in os.listdir(IMAGES_PATH) if f.endswith('.txt')]

for txt_file in tqdm(txt_files):
    image_path = os.path.join(IMAGES_PATH, txt_file.replace('.txt', '.jpg'))

    # Read image
    form = cv2.imread(image_path)

    # Read bounding box coordinates from the text file
    with open(os.path.join(IMAGES_PATH, txt_file), 'r') as f:
        lines = f.read().splitlines()

    # Parse bounding box coordinates and class from the text file
    data = [line.strip().split() for line in lines]
    class_labels = [int(row[0]) for row in data]
    bboxes = [list(map(float, row[1:])) for row in data]

    # Your augmentation code here (assuming doc_aug is defined)
    augmented = doc_aug(
        image=form,
        bboxes=bboxes,
        field_id=class_labels
    )
    
    rows = []
    boxlar = augmented['bboxes']
    for i, class_label in enumerate(augmented['field_id']):
        # Your YOLO conversion code here
        x_center, y_center, bbox_width, bbox_height = boxlar[i]

        # Create entries for each bounding box in the YOLO format without commas and parentheses
        rows.append(f"{class_label} {x_center} {y_center} {bbox_width} {bbox_height}")

    # Save augmented image
    augmented_image_path = f'{DATASET_PATH}/augmented_{txt_file.replace(".txt", ".jpg")}'
    cv2.imwrite(augmented_image_path, augmented['image'])
    
    # Save annotations in YOLO format
    with open(f'{DATASET_PATH}/augmented_{txt_file.replace(".txt", ".txt")}', 'w') as f:
        f.write('\n'.join(rows))


100%|██████████| 1/1 [00:00<00:00, 19.25it/s]


In [131]:
import os
import cv2
import pandas as pd
from tqdm import tqdm
import albumentations as A

DATASET_PATH = 'matlba/realData/valid/images'
DATASET_PATH2 = 'matlba/realData/valid/labels'
IMAGES_PATH = 'matlba/realData/valid/images'
IMAGES_PATH2 = 'matlba/realData/valid/labels'
os.makedirs(DATASET_PATH, exist_ok=True)
os.makedirs(IMAGES_PATH, exist_ok=True)

# Assuming your text files have the same name as the images but with a '.txt' extension
txt_files = [f for f in os.listdir(IMAGES_PATH2) if f.endswith('.txt')]

for txt_file in tqdm(txt_files):
    image_path = os.path.join(IMAGES_PATH, txt_file.replace('.txt', '.jpg'))

    # Read image
    form = cv2.imread(image_path)

    # Read bounding box coordinates from the text file
    with open(os.path.join(IMAGES_PATH2, txt_file), 'r') as f:
        lines = f.read().splitlines()

    # Parse bounding box coordinates and class from the text file
    data = [line.strip().split() for line in lines]
    class_labels = [int(row[0]) for row in data]
    bboxes = [list(map(float, row[1:])) for row in data]

    for _ in range(30):
        # Your augmentation code here (assuming doc_aug is defined)
        augmented = doc_aug(
            image=form,
            bboxes=bboxes,
            field_id=class_labels
        )

        rows = []
        boxlar = augmented['bboxes']
        for i, class_label in enumerate(augmented['field_id']):
            # Your YOLO conversion code here
            x_center, y_center, bbox_width, bbox_height = boxlar[i]

            # Create entries for each bounding box in the YOLO format without commas and parentheses
            rows.append(f"{class_label} {x_center} {y_center} {bbox_width} {bbox_height}")

        # Save augmented image
        augmented_image_path = f'{DATASET_PATH}/augmented_{txt_file.replace(".txt", "_{0}.jpg")}'.format(_)
        cv2.imwrite(augmented_image_path, augmented['image'])

        # Save annotations in YOLO format
        with open(f'{DATASET_PATH2}/augmented_{txt_file.replace(".txt", "_{0}.txt")}'.format(_), 'w') as f:
            f.write('\n'.join(rows))


100%|██████████| 157/157 [02:07<00:00,  1.24it/s]
